<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Supplier в C#, который будет представлять информацию о
поставщиках товаров или услуг. На основе этого класса разработать 2-3
производных класса, демонстрирующих принципы наследования и полиморфизма.
В каждом из классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;



public interface IDeliverable
{
    int DeliveryDays { get; }
    decimal CalculateDeliveryCost(double weightKg);
    string GetDeliveryInfo();
}

public interface ITrackable
{
    string TrackingNumber { get; }
    string GetTrackingStatus();
}

public interface ICertifiable
{
    string CertificateName { get; }
    bool IsCertificateValid();
}

// ============================================================
// БАЗОВЫЙ КЛАСС
// ============================================================
public class Supplier
{
    // Приватные поля
    private int _supplierId;
    private string _companyName;
    private string _productType;
    private double _rating;               

    public int SupplierId
    {
        get { return _supplierId; }
        set { _supplierId = value; }
    }

    public string CompanyName
    {
        get { return _companyName; }
        set
        {
            if (!string.IsNullOrEmpty(value))
                _companyName = value;
            else
                throw new ArgumentException("Название компании не может быть пустым!");
        }
    }

    public string ProductType
    {
        get { return _productType; }
        protected set { _productType = value; }
    }

    // ---------- НОВЫЕ АТРИБУТЫ (4) ----------
    public string Country { get; set; }                         // страна регистрации
    public DateTime RegistrationDate { get; private set; }      // дата регистрации
    public bool IsActive { get; private set; }                  // работает ли поставщик
    public double Rating                                        // рейтинг 0..5
    {
        get { return _rating; }
        private set
        {
            if (value < 0 || value > 5)
                throw new ArgumentException("Рейтинг должен быть от 0 до 5!");
            _rating = value;
        }
    }

    // Конструктор
    public Supplier(int supplierId, string companyName, string productType, string country = "Россия")
    {
        SupplierId = supplierId;
        CompanyName = companyName;
        ProductType = productType;
        Country = country;
        RegistrationDate = DateTime.Now;
        IsActive = true;
        Rating = 3.0;
    }

    // Virtual методы
    public virtual string GetCompanyInfo()
    {
        return $"ID: {SupplierId}, Компания: {CompanyName}, Тип продукции: {ProductType}, " +
               $"Страна: {Country}, Рейтинг: {Rating}";
    }

    public virtual string ProvideQuote()
    {
        return $"Котировка от {CompanyName}: стандартные условия поставки {ProductType}.";
    }

    public virtual void SendQuote()
    {
        Console.WriteLine(ProvideQuote());
    }

    public virtual string SubmitOrder(string orderDetails)
    {
        return $"Заказ '{orderDetails}' отправлен поставщику {CompanyName} (ID: {SupplierId}).";
    }

    public void LinkWithSupplier(Supplier other)
    {
        Console.WriteLine($"Поставщик {CompanyName} (ID: {SupplierId}) связан с поставщиком {other.CompanyName} (ID: {other.SupplierId})");
    }


    public void Activate() { IsActive = true; }

    public void Deactivate() { IsActive = false; }

    public void UpdateRating(double newRating) { Rating = newRating; }

    public int GetYearsOnMarket() { return DateTime.Now.Year - RegistrationDate.Year; }

    // Виртуальный: скидка зависит от суммы заказа (переопределяется в наследниках)
    public virtual decimal CalculateDiscount(decimal orderSum)
    {
        return orderSum > 100000 ? orderSum * 0.05m : 0m;
    }
}


// ПРОСТОЕ НАСЛЕДОВАНИЕ: Supplier -> Manufacturer

public class Manufacturer : Supplier
{
    private int _foundedYear;

    public int FoundedYear
    {
        get { return _foundedYear; }
        set
        {
            if (value > 1800 && value <= DateTime.Now.Year)
                _foundedYear = value;
            else
                throw new ArgumentException("Некорректный год основания!");
        }
    }

    // ---------- НОВЫЕ АТРИБУТЫ (4) ----------
    public int ProductionCapacity { get; protected set; } = 1000;    // ед. в месяц
    public int EmployeeCount { get; set; } = 50;                     // сотрудники
    public int MinOrderQuantity { get; set; } = 10;                  // мин. партия
    public decimal ProductionCostPerUnit { get; set; } = 100m;       // себестоимость

    public Manufacturer(int supplierId, string companyName, string productType, int foundedYear)
        : base(supplierId, companyName, productType)
    {
        FoundedYear = foundedYear;
    }

    public override string ProvideQuote()
    {
        return $"Котировка от производителя {CompanyName} (основан в {FoundedYear} году): " +
               $"специальные условия для {ProductType}.";
    }

    // virtual - чтобы наследники могли переопределить
    public virtual string GetManufacturerDetails()
    {
        return $"Производитель {CompanyName}, основан в {FoundedYear} году, производит {ProductType}.";
    }

    // ---------- НОВЫЕ МЕТОДЫ (4) ----------
    public int CalculateProductionDays(int quantity)
    {
        double perDay = ProductionCapacity / 30.0;
        return (int)Math.Ceiling(quantity / perDay);
    }

    public decimal CalculateOrderCost(int quantity)
    {
        if (quantity < MinOrderQuantity)
            throw new ArgumentException($"Минимальная партия: {MinOrderQuantity} ед.");
        return quantity * ProductionCostPerUnit;
    }

    public void IncreaseCapacity(int percent)
    {
        ProductionCapacity += ProductionCapacity * percent / 100;
    }

    public void HireEmployees(int count) { EmployeeCount += count; }

    // Переопределение скидки: производитель даёт больше за крупные заказы
    public override decimal CalculateDiscount(decimal orderSum)
    {
        return orderSum > 500000 ? orderSum * 0.10m : base.CalculateDiscount(orderSum);
    }

    public void SupplyToRetailer(Retailer other, string productDetails)
    {
        Console.WriteLine($"Производитель {CompanyName} поставляет товар '{productDetails}' ритейлеру {other.CompanyName}");
        Console.WriteLine($"Магазин получателя: {other.StoreLocation}");
    }
}


// МНОГОУРОВНЕВОЕ (СЛОЖНОЕ) НАСЛЕДОВАНИЕ:
// Supplier -> Manufacturer -> CertifiedManufacturer

public class CertifiedManufacturer : Manufacturer, ICertifiable
{
    // ---------- АТРИБУТЫ ----------
    public string CertificateName { get; private set; }       // из ICertifiable
    public DateTime CertificateExpiry { get; private set; }
    public int AuditsPassed { get; private set; }
    public string QualityStandard { get; set; } = "ГОСТ";

    public CertifiedManufacturer(int supplierId, string companyName, string productType,
                                 int foundedYear, string certificateName, DateTime expiry)
        : base(supplierId, companyName, productType, foundedYear)
    {
        CertificateName = certificateName;
        CertificateExpiry = expiry;
    }


    public bool IsCertificateValid() { return CertificateExpiry > DateTime.Now; }

    public void PassAudit() { AuditsPassed++; }

    public void RenewCertificate(DateTime newExpiry) { CertificateExpiry = newExpiry; }

    public override string GetManufacturerDetails()
    {
        return base.GetManufacturerDetails() +
               $" Сертификат: {CertificateName} (до {CertificateExpiry:dd.MM.yyyy}).";
    }

    public override string ProvideQuote()
    {
        return base.ProvideQuote() + $" Гарантия качества по стандарту {CertificateName}.";
    }
}


// ПРОСТОЕ НАСЛЕДОВАНИЕ: Supplier -> Retailer

public class Retailer : Supplier
{
    private string _storeLocation;

    public string StoreLocation
    {
        get { return _storeLocation; }
        set
        {
            if (!string.IsNullOrEmpty(value))
                _storeLocation = value;
            else
                throw new ArgumentException("Местоположение магазина не может быть пустым!");
        }
    }


    public int StoreCount { get; protected set; } = 1;             // число магазинов
    public decimal MarkupPercent { get; set; } = 20m;              // наценка, %
    public int ReturnPolicyDays { get; set; } = 14;                // срок возврата
    public bool HasDelivery { get; set; } = false;                 // есть ли доставка

    public Retailer(int supplierId, string companyName, string productType, string storeLocation)
        : base(supplierId, companyName, productType)
    {
        StoreLocation = storeLocation;
    }

    public override string SubmitOrder(string orderDetails)
    {
        return $"Заказ '{orderDetails}' отправлен ритейлеру {CompanyName} " +
               $"(магазин: {StoreLocation}, ID: {SupplierId}).";
    }

    public string GetRetailerInfo()
    {
        return $"Ритейлер {CompanyName}, расположение: {StoreLocation}, ассортимент: {ProductType}.";
    }

    // ---------- НОВЫЕ МЕТОДЫ (4) ----------
    public decimal CalculateRetailPrice(decimal wholesalePrice)
    {
        return wholesalePrice * (1 + MarkupPercent / 100m);
    }

    public bool CanReturn(int daysSincePurchase) { return daysSincePurchase <= ReturnPolicyDays; }

    public void OpenNewStore() { StoreCount++; }

    public void SetMarkup(decimal percent)
    {
        if (percent < 0) throw new ArgumentException("Наценка не может быть отрицательной!");
        MarkupPercent = percent;
    }

    public void OrderFromManufacturer(Manufacturer other, string orderDetails)
    {
        Console.WriteLine($"Ритейлер {CompanyName} оформляет заказ '{orderDetails}' у производителя {other.CompanyName}");
        Console.WriteLine(other.SubmitOrder(orderDetails));
    }

    public void OrderFromImporter(Importer other, string orderDetails)
    {
        Console.WriteLine($"Ритейлер {CompanyName} оформляет заказ '{orderDetails}' у импортёра {other.CompanyName}");
        Console.WriteLine(other.SubmitOrder(orderDetails));
    }
}


// МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ:
// класс Retailer + интерфейсы IDeliverable и ITrackable

public class OnlineRetailer : Retailer, IDeliverable, ITrackable
{
    // ---------- АТРИБУТЫ ----------
    public string WebsiteUrl { get; set; }
    public int DeliveryDays { get; set; } = 3;                // из IDeliverable
    public string TrackingNumber { get; private set; } = "—"; // из ITrackable
    public decimal DeliveryRatePerKg { get; set; } = 40m;

    public OnlineRetailer(int supplierId, string companyName, string productType,
                          string storeLocation, string websiteUrl)
        : base(supplierId, companyName, productType, storeLocation)
    {
        WebsiteUrl = websiteUrl;
        HasDelivery = true;
    }

    // ---------- Реализация IDeliverable ----------
    public decimal CalculateDeliveryCost(double weightKg)
    {
        return 200m + (decimal)weightKg * DeliveryRatePerKg;
    }

    public string GetDeliveryInfo()
    {
        return $"{CompanyName} ({WebsiteUrl}): доставка за {DeliveryDays} дн.";
    }

    // ---------- Реализация ITrackable ----------
    public string GetTrackingStatus()
    {
        return TrackingNumber == "—"
            ? "Трек-номер ещё не присвоен."
            : $"Заказ {TrackingNumber} передан в доставку.";
    }

    public void GenerateTrackingNumber()
    {
        TrackingNumber = $"RU{SupplierId:D3}{DateTime.Now:ddMMyy}{new Random().Next(1000, 9999)}";
    }
}

// ПРОСТОЕ НАСЛЕДОВАНИЕ: Supplier -> Importer (+ интерфейс)

public class Importer : Supplier, IDeliverable
{
    private string _originCountry;

    public string OriginCountry
    {
        get { return _originCountry; }
        set { _originCountry = value; }
    }

    // ---------- НОВЫЕ АТРИБУТЫ (4) ----------
    public decimal CustomsDutyPercent { get; set; } = 10m;       // таможенная пошлина, %
    public int DeliveryDays { get; set; } = 21;                  // срок доставки (IDeliverable)
    public string Currency { get; set; } = "USD";                // валюта расчётов
    public string CustomsBroker { get; set; } = "Не назначен";   // таможенный брокер

    public Importer(int supplierId, string companyName, string productType, string originCountry)
        : base(supplierId, companyName, productType, "Россия")
    {
        OriginCountry = originCountry;
    }

    public override string GetCompanyInfo()
    {
        return base.GetCompanyInfo() + $", Страна происхождения: {OriginCountry}";
    }

    public string GetImportDetails()
    {
        return $"Импортер {CompanyName} поставляет {ProductType} из {OriginCountry}.";
    }

    // ---------- НОВЫЕ МЕТОДЫ (4) ----------
    public decimal CalculateCustomsDuty(decimal goodsValue)
    {
        return goodsValue * CustomsDutyPercent / 100m;
    }

    public decimal GetTotalImportCost(decimal goodsValue)
    {
        return goodsValue + CalculateCustomsDuty(goodsValue);
    }

    public void ChangeBroker(string newBroker) { CustomsBroker = newBroker; }

    // Реализация IDeliverable
    public decimal CalculateDeliveryCost(double weightKg) { return 1500m + (decimal)weightKg * 150m; }

    public string GetDeliveryInfo()
    {
        return $"Импортёр {CompanyName}: доставка из {OriginCountry} за {DeliveryDays} дн., брокер: {CustomsBroker}.";
    }

    public void SupplyToManufacturer(Manufacturer other, string componentDetails)
    {
        Console.WriteLine($"Импортёр {CompanyName} поставляет компоненты '{componentDetails}' производителю {other.CompanyName}");
        Console.WriteLine($"Страна происхождения: {OriginCountry}");
    }
}
Manufacturer myManufacturer = new Manufacturer(1, "ООО 'ТехноСтрой'", "Строительные материалы", 2005);
Retailer myRetailer = new Retailer(2, "ООО 'Мир Электроники'", "Бытовая техника", "г. Москва, ул. Тверская, 15");
Importer myImporter = new Importer(3, "ООО 'Глобал Трейд'", "Электронные компоненты", "Китай");
Manufacturer myManufacturer2 = new Manufacturer(4, "ООО 'МеталлПром'", "Металлопрокат", 1998);

// Многоуровневое наследование: Supplier -> Manufacturer -> CertifiedManufacturer
CertifiedManufacturer certified = new CertifiedManufacturer(
    6, "ООО 'КачествоПлюс'", "Бетон", 2010, "ISO 9001", new DateTime(2028, 12, 31));

// Множественное наследование (класс + интерфейсы): Retailer + IDeliverable + ITrackable
OnlineRetailer shop = new OnlineRetailer(
    7, "ООО 'ИнтернетМаркет'", "Электроника", "г. Казань, ул. Баумана, 5", "https://market.example.ru");

Console.WriteLine("=== Базовые методы ===");
myManufacturer.SendQuote();
myRetailer.SendQuote();
myImporter.SendQuote();

Console.WriteLine("\n=== Взаимодействие ===");
myManufacturer.LinkWithSupplier(myRetailer);
myManufacturer.SupplyToRetailer(myRetailer, "партия строительных блоков");
myImporter.SupplyToManufacturer(myManufacturer, "микросхемы, 500 шт.");
myRetailer.OrderFromManufacturer(myManufacturer2, "500 ед. цемента");
myRetailer.OrderFromImporter(myImporter, "партия смартфонов");

Console.WriteLine("\n=== Новые методы базового класса ===");
myManufacturer.UpdateRating(4.5);
Console.WriteLine($"Рейтинг: {myManufacturer.Rating}, лет на рынке: {myManufacturer.GetYearsOnMarket()}");
Console.WriteLine($"Скидка (базовая, 150000): {myRetailer.CalculateDiscount(150000)}");
myRetailer.Deactivate();
Console.WriteLine($"Ритейлер активен: {myRetailer.IsActive}");
myRetailer.Activate();

Console.WriteLine("\n=== Manufacturer ===");
Console.WriteLine($"Дней на выпуск 3000 ед.: {myManufacturer.CalculateProductionDays(3000)}");
Console.WriteLine($"Стоимость заказа 3000 ед.: {myManufacturer.CalculateOrderCost(3000)} руб.");
Console.WriteLine($"Скидка производителя (600000): {myManufacturer.CalculateDiscount(600000)}");
myManufacturer.IncreaseCapacity(20);
Console.WriteLine($"Новая мощность: {myManufacturer.ProductionCapacity} ед./мес.");

Console.WriteLine("\n=== Retailer ===");
Console.WriteLine($"Розничная цена (опт 1000): {myRetailer.CalculateRetailPrice(1000)} руб.");
Console.WriteLine($"Возврат через 10 дней возможен: {myRetailer.CanReturn(10)}");
myRetailer.OpenNewStore();
Console.WriteLine($"Магазинов: {myRetailer.StoreCount}");

Console.WriteLine("\n=== Importer (+ IDeliverable) ===");
Console.WriteLine($"Пошлина с 100000: {myImporter.CalculateCustomsDuty(100000)} руб.");
Console.WriteLine($"Полная стоимость импорта: {myImporter.GetTotalImportCost(100000)} руб.");
Console.WriteLine(myImporter.GetDeliveryInfo());

Console.WriteLine("\n=== Многоуровневое наследование ===");
Console.WriteLine(certified.GetManufacturerDetails());
certified.PassAudit();
Console.WriteLine($"Сертификат действителен: {certified.IsCertificateValid()}, аудитов: {certified.AuditsPassed}");
Console.WriteLine(certified.ProvideQuote());

Console.WriteLine("\n=== Множественное наследование (интерфейсы) ===");
Console.WriteLine(shop.GetDeliveryInfo());
Console.WriteLine($"Доставка 12.5 кг: {shop.CalculateDeliveryCost(12.5)} руб.");
shop.GenerateTrackingNumber();
Console.WriteLine(shop.GetTrackingStatus());
IDeliverable deliverable = shop;   // один объект - разные "роли"
ITrackable trackable = shop;
Console.WriteLine($"Через интерфейсы: {deliverable.DeliveryDays} дн., трек {trackable.TrackingNumber}");

Console.WriteLine("\n=== Информация о компаниях ===");
Console.WriteLine(myManufacturer.GetCompanyInfo());
Console.WriteLine(myRetailer.GetCompanyInfo());
Console.WriteLine(myImporter.GetCompanyInfo());


=== Базовые методы ===
Котировка от производителя ООО 'ТехноСтрой' (основан в 2005 году): специальные условия для Строительные материалы.
Котировка от ООО 'Мир Электроники': стандартные условия поставки Бытовая техника.
Котировка от ООО 'Глобал Трейд': стандартные условия поставки Электронные компоненты.

=== Взаимодействие ===
Поставщик ООО 'ТехноСтрой' (ID: 1) связан с поставщиком ООО 'Мир Электроники' (ID: 2)
Производитель ООО 'ТехноСтрой' поставляет товар 'партия строительных блоков' ритейлеру ООО 'Мир Электроники'
Магазин получателя: г. Москва, ул. Тверская, 15
Импортёр ООО 'Глобал Трейд' поставляет компоненты 'микросхемы, 500 шт.' производителю ООО 'ТехноСтрой'
Страна происхождения: Китай
Ритейлер ООО 'Мир Электроники' оформляет заказ '500 ед. цемента' у производителя ООО 'МеталлПром'
Заказ '500 ед. цемента' отправлен поставщику ООО 'МеталлПром' (ID: 4).
Ритейлер ООО 'Мир Электроники' оформляет заказ 'партия смартфонов' у импортёра ООО 'Глобал Трейд'
Заказ 'партия смартфонов' отп